# nn-parameter-wrap — worked example 1: Parameter vs raw tensor visibility

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `nn-parameter-wrap`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

Wrapping a tensor in `nn.Parameter` is what makes it visible to `.parameters()`, `.state_dict()`, and the optimizer. A raw tensor stored as an attribute is invisible to all of those, even though it works fine in `forward`.

## Worked solution

We build two near-identical modules, each storing a single `(3,)` tensor as `weight`. `RawMod` stores it as a plain `t.randn(3)`; `WrappedMod` wraps the same draw in `nn.Parameter`. We reseed immediately before each draw so the numeric values match and only the wrapping differs. Both have working `forward` methods returning `x * self.weight`. The decisive observation is registration: `RawMod` reports zero parameters and an empty state dict, while `WrappedMod` reports exactly one parameter named `weight`. We print both counts to make the contrast explicit; the only code difference is the `nn.Parameter` wrap.

In [ ]:
import torch as t
import torch.nn as nn

class RawMod(nn.Module):
    def __init__(self):
        super().__init__()
        t.manual_seed(7)
        self.weight = t.randn(3)  # raw -> invisible
    def forward(self, x):
        return x * self.weight

class WrappedMod(nn.Module):
    def __init__(self):
        super().__init__()
        t.manual_seed(7)
        self.weight = nn.Parameter(t.randn(3))  # wrapped -> registered
    def forward(self, x):
        return x * self.weight

raw, wrapped = RawMod(), WrappedMod()
print('raw params:', len(list(raw.parameters())))
print('wrapped params:', len(list(wrapped.parameters())))
print('wrapped state_dict keys:', list(wrapped.state_dict().keys()))